# ML-07 — Baseline Action Score and Top-10 Review

[Open in Colab](https://colab.research.google.com/github/mhdaamiryo/fly-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane:** Content Refresh / Search Intelligence ranking  
**Data window:** March 1–31, 2026  
**Decision point:** after March 31, 2026

This notebook checks two observable signals, encodes one transparent rule, writes a ranked queue, and reviews its ten highest-priority rows. The queue is descriptive decision-support, not a claim that a refresh will cause recovery.

**Privacy:** pseudonymous client/content keys are used only for grouping. They are never displayed or written to the output CSV. No URLs, client names, or private queries are selected.

**Run:** add your Hugging Face READ token to Colab Secrets as HF_TOKEN, then Runtime → Run all. Never paste a token into a cell or commit it.

## 1. Setup and connect

This uses the authenticated DuckDB + Hugging Face workflow from ML-04. It reads only the March 2026 partition. No June sample partition, future window, product flag, or label-derived field is used.

In [4]:
%pip -q install duckdb huggingface_hub pandas numpy

In [5]:
import os
import duckdb
import numpy as np
import pandas as pd
from IPython.display import display
from pathlib import Path

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Add your Hugging Face READ token as a Colab Secret named HF_TOKEN.")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
print("Authenticated warehouse connection ready. Reading March 2026 only.")

Authenticated warehouse connection ready. Reading March 2026 only.


## 2. Build the March search snapshot

One row represents one pseudonymous client-content pair aggregated over March. Position is averaged over non-zero daily position values; zero means no position observation. CTR is total clicks divided by total impressions, not an average of daily CTRs.

These are March-close descriptive signals, not future-window predictors.

In [6]:
march_sql = f"""
    SELECT client_hash_id, content_hash_id,
           SUM(COALESCE(gsc_impressions, 0)) AS impressions,
           SUM(COALESCE(gsc_clicks, 0)) AS clicks,
           AVG(NULLIF(gsc_avg_position, 0)) AS avg_position,
           COUNT(DISTINCT report_date) AS observed_days
    FROM {MARCH}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
    GROUP BY client_hash_id, content_hash_id
"""
df = con.sql(march_sql).df()
for c in ["impressions", "clicks"]:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
df["avg_position"] = pd.to_numeric(df["avg_position"], errors="coerce")
df["ctr_pct"] = np.where(df["impressions"] > 0, 100 * df["clicks"] / df["impressions"], np.nan)
assert len(df) > 0, "March snapshot is empty; check warehouse access and partition path."
assert (df["clicks"] <= df["impressions"]).all(), "Unexpected clicks > impressions; inspect source metrics."
print(f"March client-content rows: {len(df):,}")
print(f"Rows with a usable average position: {df['avg_position'].notna().sum():,}")
print(f"Total March impressions: {df['impressions'].sum():,.0f}")
print(f"Total March clicks: {df['clicks'].sum():,.0f}")
display(df[["impressions","clicks","ctr_pct","avg_position","observed_days"]].describe().T)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March client-content rows: 331,437
Rows with a usable average position: 175,304
Total March impressions: 280,657,589
Total March clicks: 821,832


,count,mean,std,min,25%,50%,75%,max
impressions,331437.0,846.790156,4044.514753,0.000000,0.0,2.0,216.000000,617124.0
clicks,331437.0,2.479602,19.651282,0.000000,0.0,0.0,0.000000,5668.0
ctr_pct,176738.0,0.459397,3.775992,0.000000,0.0,0.0,0.215796,100.0
avg_position,175304.0,17.050555,18.333942,0.101639,5.5,9.0,22.000000,309.0
observed_days,331437.0,29.693058,4.738734,1.000000,31.0,31.0,31.000000,31.0


### Signal check A — CTR relative to search position (flag-linked)

**Question:** does median CTR generally decrease as average position gets worse? This checks the relationship behind CTR-fix logic. The bucket table prints n, median CTR, the 25th-percentile CTR, and median impressions for every position band.

**Verdict rule:** compare median CTR at positions 1–10 with positions 11–50. Higher at 1–10 = CONFIRMED; lower = OPPOSITE; insufficient or equal evidence = MIXED. This describes this observed slice only.

In [7]:
position_bins = [0, 3, 10, 20, 50, np.inf]
position_labels = ["1-3", "4-10", "11-20", "21-50", "50+"]
df["position_band"] = pd.cut(df["avg_position"], bins=position_bins, labels=position_labels)
pos = df[df["avg_position"].notna() & df["ctr_pct"].notna()].copy()
position_table = (
    pos.groupby("position_band", observed=False)
       .agg(n=("ctr_pct", "size"),
            median_ctr_pct=("ctr_pct", "median"),
            p25_ctr_pct=("ctr_pct", lambda s: s.quantile(0.25)),
            median_impressions=("impressions", "median"))
       .reset_index()
)
display(position_table)
top10_bands = pos[pos["avg_position"] <= 10]["ctr_pct"]
lower_bands = pos[(pos["avg_position"] > 10) & (pos["avg_position"] <= 50)]["ctr_pct"]
if len(top10_bands) == 0 or len(lower_bands) == 0:
    ctr_verdict, ctr_reason = "MIXED", "One or both comparison groups have no observations."
else:
    top10_median, lower_median = float(top10_bands.median()), float(lower_bands.median())
    ctr_verdict = "CONFIRMED" if top10_median > lower_median else ("OPPOSITE" if top10_median < lower_median else "MIXED")
    ctr_reason = f"Median CTR at positions 1–10 = {top10_median:.3f}%; positions 11–50 = {lower_median:.3f}%."
print(f"CTR-vs-position verdict: {ctr_verdict}. {ctr_reason}")

,position_band,n,median_ctr_pct,p25_ctr_pct,median_impressions
0,1-3,13136,0.096246,0.0,654.0
1,4-10,81619,0.000000,0.0,204.0
2,11-20,32548,0.000000,0.0,256.0
3,21-50,34783,0.000000,0.0,165.0
4,50+,13218,0.000000,0.0,45.0


CTR-vs-position verdict: MIXED. Median CTR at positions 1–10 = 0.000%; positions 11–50 = 0.000%.


### Signal check B — search volume (quick-win flag-linked)

**Question:** how much observed search demand sits on pages with at least 500 March impressions? Higher-volume pages can offer more potential click impact, checking the volume signal behind quick-win prioritisation.

The table reports n, total impressions, median clicks, and share of all impressions.

**Verdict rule:** at least 50% of all impressions in pages with 500+ impressions = CONFIRMED; under 25% = OPPOSITE; otherwise = MIXED. These are declared review thresholds, not FlyRank's proprietary thresholds.

In [8]:
volume_bins = [-0.001, 99.999, 499.999, 2999.999, np.inf]
volume_labels = ["0-99", "100-499", "500-2,999", "3,000+"]
df["volume_band"] = pd.cut(df["impressions"], bins=volume_bins, labels=volume_labels, include_lowest=True)
volume_table = (
    df.groupby("volume_band", observed=False)
      .agg(n=("impressions", "size"),
           total_impressions=("impressions", "sum"),
           median_clicks=("clicks", "median"))
      .reset_index()
)
all_impressions = float(df["impressions"].sum())
volume_table["share_of_impressions_pct"] = np.where(
    all_impressions > 0, 100 * volume_table["total_impressions"] / all_impressions, 0
)
display(volume_table)
high_volume_share = float(df.loc[df["impressions"] >= 500, "impressions"].sum() / all_impressions) if all_impressions > 0 else 0.0
volume_verdict = "CONFIRMED" if high_volume_share >= 0.50 else ("OPPOSITE" if high_volume_share < 0.25 else "MIXED")
print(f"Volume verdict: {volume_verdict}. Pages with 500+ impressions account for {high_volume_share:.1%} of March impressions.")

,volume_band,n,total_impressions,median_clicks,share_of_impressions_pct
0,0-99,229996,1861506.0,0.0,0.663266
1,100-499,39517,9879964.0,0.0,3.520291
2,"500-2,999",39767,52757859.0,2.0,18.797945
3,"3,000+",22157,216158260.0,15.0,77.018498


Volume verdict: CONFIRMED. Pages with 500+ impressions account for 95.8% of March impressions.


## 3. One baseline rule and reason code

**Rule:** review a page's search snippet when it has at least 500 March impressions, average position in the top 20, and CTR below the 25th-percentile CTR of pages in the same position band. Rank eligible pages by estimated missed clicks:

priority_score = impressions × (position-band p25 CTR − page CTR) / 100

This is a transparent prioritisation heuristic, not a fitted model or causal-lift estimate.

- **One reason code:** visible_low_ctr
- **Action label:** review_search_snippet
- **Not selected:** no_action (no reason code)

The within-position benchmark avoids treating naturally lower CTR at worse positions as a snippet problem. The 500-impression floor limits very low-volume noise.

In [11]:
# Calculate position-band benchmarks using pages with positive CTR
positive_pos = pos[pos["ctr_pct"] > 0].copy()

position_benchmark = (
    positive_pos.groupby("position_band", observed=False)
    .agg(
        positive_ctr_pages=("ctr_pct", "size"),
        p25_positive_ctr=("ctr_pct", lambda x: x.quantile(0.25))
    )
)

display(position_benchmark.reset_index())

# Map benchmarks back to all pages
df["position_p25_ctr_pct"] = (
    df["position_band"].map(position_benchmark["p25_positive_ctr"]).astype(float)
)

# Calculate the CTR gap
df["ctr_gap_pp"] = (
    df["position_p25_ctr_pct"] - df["ctr_pct"]
).clip(lower=0)

# Eligibility rule
eligible = (
    df["impressions"].ge(500)
    & df["avg_position"].le(20)
    & df["position_p25_ctr_pct"].notna()
    & df["ctr_gap_pp"].gt(0)
)

# Calculate priority score
df["priority_score"] = np.where(
    eligible,
    df["impressions"] * df["ctr_gap_pp"] / 100,
    0.0
)

# One reason code and one action
df["reason_code"] = np.where(
    eligible, "visible_low_ctr", ""
)

df["action"] = np.where(
    eligible, "review_search_snippet", "no_action"
)

# Rank all pages
queue = df.sort_values(
    ["priority_score", "impressions"],
    ascending=[False, False]
).reset_index(drop=True)

queue.insert(0, "rank", np.arange(1, len(queue) + 1))

flagged = queue[queue["priority_score"] > 0].copy()

print(f"All ranked rows: {len(queue):,}")
print(f"Rows selected for snippet review: {len(flagged):,}")
print(f"Highest priority score: {queue['priority_score'].max():,.3f}")

assert len(flagged) >= 10, "Fewer than 10 eligible rows; inspect the benchmark."

,position_band,positive_ctr_pages,p25_positive_ctr
0,1-3,7322,0.173528
1,4-10,36958,0.171404
2,11-20,13295,0.164508
3,21-50,10599,0.090423
4,50+,606,0.226118


All ranked rows: 331,437
Rows selected for snippet review: 21,666
Highest priority score: 340.069


### Write the ranked queue

The CSV contains rank, score, action/reason, and aggregate search metrics. It deliberately excludes client/content keys, URLs, names, and query text. The output directory is created automatically. The assignment's CI policy keeps CSV data outputs out of Git.

In [12]:
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)
csv_path = output_dir / "baseline_action_score.csv"
output_columns = [
    "rank", "priority_score", "reason_code", "action",
    "impressions", "clicks", "ctr_pct", "avg_position",
    "position_p25_ctr_pct", "ctr_gap_pp", "observed_days"
]
queue[output_columns].to_csv(csv_path, index=False)
print(f"Ranked queue written: {csv_path}")
print(f"CSV rows: {len(queue):,}; selected actions: {(queue['action'] == 'review_search_snippet').sum():,}")
display(queue[output_columns].head(10).round(3))

Ranked queue written: work/outputs/baseline_action_score.csv
CSV rows: 331,437; selected actions: 21,666


,rank,priority_score,reason_code,action,impressions,clicks,ctr_pct,avg_position,position_p25_ctr_pct,ctr_gap_pp,observed_days
0,1,340.069,visible_low_ctr,review_search_snippet,212404.0,24.0,0.011,7.347,0.171,0.160,31
1,2,230.368,visible_low_ctr,review_search_snippet,134984.0,1.0,0.001,4.546,0.171,0.171,31
2,3,211.670,visible_low_ctr,review_search_snippet,124075.0,1.0,0.001,9.385,0.171,0.171,31
3,4,202.140,visible_low_ctr,review_search_snippet,143019.0,43.0,0.030,3.219,0.171,0.141,31
4,5,169.403,visible_low_ctr,review_search_snippet,107584.0,15.0,0.014,9.536,0.171,0.157,31
5,6,149.119,visible_low_ctr,review_search_snippet,89332.0,4.0,0.004,7.786,0.171,0.167,31
6,7,144.270,visible_low_ctr,review_search_snippet,132593.0,83.0,0.063,5.789,0.171,0.109,31
7,8,137.616,visible_low_ctr,review_search_snippet,83788.0,6.0,0.007,7.289,0.171,0.164,31
8,9,136.913,visible_low_ctr,review_search_snippet,83834.0,1.0,0.001,11.967,0.165,0.163,31
9,10,130.196,visible_low_ctr,review_search_snippet,82376.0,11.0,0.013,7.794,0.171,0.158,31


## 4. Top-10 review — skeptical read

Each selected row receives an action, the measured reason it ranked, a confidence note, and a concrete condition that could make the recommendation wrong. Rank is used instead of pseudonymous IDs so no client/content key is exposed.

These are first-pass analyst review notes generated from observed metrics. They do not replace checking the page's query mix and live search results before taking action.

In [13]:
top10 = flagged.head(10).copy()
review_rows = []
for _, row in top10.iterrows():
    why = (
        f"CTR {row['ctr_pct']:.3f}% is {row['ctr_gap_pp']:.3f} percentage points "
        f"below its position-band p25 ({row['position_p25_ctr_pct']:.3f}%) "
        f"across {row['impressions']:,.0f} impressions; score {row['priority_score']:.3f}."
    )
    confidence = (
        "Moderate: clears 500-impression floor, but volume is still limited."
        if row["impressions"] < 1000 else
        "Moderate: meaningful observed volume; query-level context still needed."
    )
    wrong_if = (
        "Wrong if query intent, brand mix, or SERP features naturally produce a lower CTR "
        "than the cohort benchmark; aggregate position may hide very different queries."
    )
    review_rows.append({
        "rank": int(row["rank"]), "action": row["action"],
        "why_it_is_here": why, "confidence": confidence,
        "what_would_make_it_wrong": wrong_if
    })
top10_review = pd.DataFrame(review_rows)
assert len(top10_review) == 10
display(top10_review.to_string(index=False))
print("Inspect query intent and current SERP presentation before approving any snippet change.")

' rank                action                                                                                                        why_it_is_here                                                              confidence                                                                                                                                        what_would_make_it_wrong\n    1 review_search_snippet CTR 0.011% is 0.160 percentage points below its position-band p25 (0.171%) across 212,404 impressions; score 340.069. Moderate: meaningful observed volume; query-level context still needed. Wrong if query intent, brand mix, or SERP features naturally produce a lower CTR than the cohort benchmark; aggregate position may hide very different queries.\n    2 review_search_snippet CTR 0.001% is 0.171 percentage points below its position-band p25 (0.171%) across 134,984 impressions; score 230.368. Moderate: meaningful observed volume; query-level context still needed. Wrong if query intent, 

Inspect query intent and current SERP presentation before approving any snippet change.


## 5. Weak picks and leakage check

The weakest three within the top ten are the rows with the smallest positive priority scores. They still meet the rule, but their estimated click gaps are closest to the selection boundary.

**Limitations:** the position benchmark is calculated from this March cohort; aggregate position can conceal query-level variation; CTR can be affected by intent, brand demand, snippets, and SERP features. The score is a queueing heuristic, not a prediction of which page will recover.

In [14]:
weak_picks = top10.sort_values("priority_score", ascending=True).head(3).copy()
weak_picks["why_weak"] = weak_picks.apply(
    lambda r: (
        f"Smallest top-10 score group: estimated gap is only {r['ctr_gap_pp']:.3f} pp "
        f"at {r['impressions']:,.0f} impressions; confirm query mix before action."
    ), axis=1
)
display(weak_picks[["priority_score","impressions","ctr_pct","position_p25_ctr_pct","ctr_gap_pp","why_weak"]].round(3))

allowed_rule_inputs = {
    "impressions", "clicks", "avg_position", "ctr_pct", "position_band",
    "position_p25_ctr_pct", "ctr_gap_pp"
}
forbidden_terms = ("label", "trend", "future", "declin", "june", "ga4", "client_hash", "content_hash")
assert not any(any(term in col.lower() for term in forbidden_terms) for col in allowed_rule_inputs)
assert "is_declining_label" not in df.columns
assert "trend_pct" not in df.columns
assert "trend_direction" not in df.columns
assert "ga4_data_available" not in df.columns
assert "client_hash_id" not in output_columns and "content_hash_id" not in output_columns
print("Leakage guard passed: March search observations only; no label, future window, GA4, or IDs.")
print("No precision@K is claimed: this action-ranking task has no independently adjudicated refresh-success label in this slice.")

,priority_score,impressions,ctr_pct,position_p25_ctr_pct,ctr_gap_pp,why_weak
9,130.196,82376.0,0.013,0.171,0.158,Smallest top-10 score group: estimated gap is ...
8,136.913,83834.0,0.001,0.165,0.163,Smallest top-10 score group: estimated gap is ...
7,137.616,83788.0,0.007,0.171,0.164,Smallest top-10 score group: estimated gap is ...


Leakage guard passed: March search observations only; no label, future window, GA4, or IDs.
No precision@K is claimed: this action-ranking task has no independently adjudicated refresh-success label in this slice.


## 6. Self-check

- [x] Two flag-linked signal checks are implemented with bucket tables and n.
- [x] Verdicts are calculated from authenticated March data when run.
- [x] One transparent score, one reason code, and an action label are encoded.
- [x] Ranked queue writes to work/outputs/baseline_action_score.csv.
- [x] Ten rows receive action, reason, confidence, and a skeptical failure condition.
- [x] Weak picks are surfaced.
- [x] Leakage guard excludes future windows, labels, GA4, and identifiers from the rule.
- [ ] Run top-to-bottom in Colab with HF_TOKEN.
- [ ] Inspect real tables and top-10 output; revise any notes that do not fit observed rows.
- [ ] Save the executed notebook back to this repository.

**Interpretation boundary:** this is a March-close, search-only review queue. It is not evidence that snippet edits cause more clicks. The next model must use a properly aligned prediction target and be evaluated against an honestly frozen baseline.

**Submission:** commit the executed notebook, then submit the repository URL on the ML-07 card. The generated CSV remains out of Git by design.